# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIRˆ²](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the `mlcroissant` library, following the Croissant specification for reproducible ML-ready data.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if it's not already installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset Croissant metadata and associated records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic metadata via object attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview

Let's review available record sets and their associated fields. All references are given by their Croissant `@id`.

- We'll first enumerate all record sets in the dataset schema.
- Then for each record set, we'll print its `@id` and list field `@id`s.

In [ ]:
from pprint import pprint

# Explore all record sets in the dataset metadata
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<no name>')}")
    print(f"  Description: {rs.get('description', '<no description>')}")
    fields = rs.get('fields', [])
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict) and '@id' in fld:
            print(f"    Field @id: {fld['@id']}")
        elif isinstance(fld, str):
            print(f"    Field @id: {fld}")
    print()

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s shown above as references for selection.

In [ ]:
# List of record set @id's from the overview above - adjust if there are multiple record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = dict()

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")

# Pick the first record set for downstream demo
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    print(f"\nExample records from RecordSet {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available.")

## 4. Exploratory Data Analysis (EDA)

We now perform basic EDA steps on the main record set. We'll dynamically select a likely numeric field by its `@id`, filter the DataFrame, normalize the field, and attempt to group by a categorical field.

In [ ]:
# Choose a main record set DataFrame
if not main_record_set_id:
    print("No record sets available for EDA.")
else:
    df = dataframes[main_record_set_id]
    print(f"Columns: {list(df.columns)}\n")
    
    # Attempt to guess a numeric field to demonstrate filtering/normalization
    # We'll look for a field with integer or float dtype and sensible values
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().sum() > 0:
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No numeric field found for EDA.\n")
    else:
        print(f"Using numeric field: {numeric_field_id} (referenced by @id)")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 10

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to guess a groupable field (non-numeric, low distinct count, not null)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                nunique = df[col].nunique(dropna=True)
                if 2 <= nunique <= min(10, len(df)//5):
                    group_field_id = col
                    break
        if group_field_id and group_field_id in filtered_df.columns:
            print(f"\nGrouping filtered data by {group_field_id} (@id):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
            display(grouped_df.head())
        else:
            print("No suitable groupable categorical field identified for grouping.\n")

## 5. Visualization

Here we visualize the distribution of the chosen numeric field, and its relation to a group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:

- Load a Croissant-defined dataset using the `mlcroissant` library;
- Reference all dataset elements—record sets, fields and columns—by their `@id`;
- Convert record sets to pandas DataFrames for flexible analysis;
- Perform basic data exploration, transformation, and visualization;

This workflow enables reproducible, schema-driven analytics on FAIRˆ² datasets for clinical and molecular oncology research. For deeper analyses, continue referencing metadata and fields by their unique `@id` as shown.